# SQL query from table names - Continued

In [ ]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

GEMINI_API_KEY = ''

## The old Prompt

In [11]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [12]:
context = [ {'role':'system', 'content':"""
 CREATE SEVERAL employees (
    ID_usr int,
    name varchar)
"""} ]



In [13]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
Question: Who is the richest employee?
 
"""
})

In [14]:
import google.generativeai as genai

# Function to call the model.
def return_CCRMSQL(user_message, context):
    # Configure the Gemini API with your key.
    # This assumes you have a variable named GEMINI_API_KEY containing your key string.
    genai.configure(api_key=GEMINI_API_KEY)
    
    # Initialize the model
    model = genai.GenerativeModel('gemini-2.0-flash')

    # Convert the OpenAI-style context (list of dicts) into a text prompt for Gemini
    prompt_text = ""
    for message in context:
        role = message.get('role', '')
        content = message.get('content', '')
        # Format as "role: content"
        prompt_text += f"{role}: {content}\n"

    # Append the current user message
    prompt_text += f"user: question: {user_message}"

    # Generate the response
    response = model.generate_content(
        prompt_text,
        generation_config=genai.types.GenerationConfig(temperature=0)
    )

    return response.text



## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [15]:
#new
context_user = context.copy()
print(return_CCRMSQL("most rich person", context_user))

```sqlite
SELECT name FROM employees ORDER BY ID_usr DESC LIMIT 1;
```


In [21]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("most rich", old_context_user))

ResourceExhausted: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.

In [19]:
#new
print(return_CCRMSQL("the first abc person", context_user))

ResourceExhausted: 429 Resource exhausted. Please try again later. Please refer to https://cloud.google.com/vertex-ai/generative-ai/docs/error-code-429 for more details.

In [ ]:
#old
print(return_CCRMSQL("how are you", old_context_user))

Okay, I understand the database schema. Now, please provide your query so I can generate the appropriate SQL command for you.



# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?